In [142]:
import pandas as pd
from sqlalchemy import create_engine
import urllib
import ast
from datetime import timedelta

In [ ]:
params = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=181.57.189.150,1433;"
    "DATABASE=master;"
    "UID=sa;"
    "PWD=P@ssw0rd*;"
)

# Crear el motor de conexión
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

In [144]:
query = """
SELECT DISTINCT v.Cliente,
    c.Nombres + ' ' + c.Apellidos AS Nombre,
    c.Email AS Email,
    c.Celular AS Celular,
    v.Fecha AS Fecha_Venta,
    v.Canal,
    SUM(v.Venta_Neta) AS Venta
FROM Ventas_Comerssia.dbo.View_Contacto_Clientes c
INNER JOIN Ventas_Comerssia.dbo.Ventas_Unificadas v ON c.Cliente = v.Cliente
WHERE v.fecha >= '2025-07-19'
    AND v.Fecha <= '2025-07-19'
    AND v.Venta_Neta > 0
    AND Canal IN ('Shopify', 'CHATCENTER WEB')
GROUP BY  v.Cliente,
    c.Nombres + ' ' + c.Apellidos,
    c.Email,
    c.Celular,
    v.Fecha,
    v.Canal
"""

# Ejecutar y cargar en DataFrame
df_ventas = pd.read_sql(query, engine)
df_ventas.head()

,Cliente,Nombre,Email,Celular,Fecha_Venta,Canal,Venta
0,C,JULIE ORELLANOS,nayi39@gmail.com,3044041231,2025-07-19,Shopify,2916051.07
1,C1000180863,VALENTINA ROA,mariavaro28@gmail.com,3013758174,2025-07-19,Shopify,180244.35
2,C1000396640,CAROLINA RESTREPO,carolina0600@gmail.com,3108267882,2025-07-19,Shopify,141170.40
3,C1001359727,CRISTINA CORTES SUTA,cristinacortes97@gmail.com,3012851765,2025-07-19,Shopify,144951.75
4,C1001369188,MARIA JOSE GUTIERREZ,majose0827@gmail.com,3124667861,2025-07-19,CHATCENTER WEB,297256.12


In [145]:
# #consulta segmentada 

# query = """
# SELECT DISTINCT v.Cliente,
#     c.Nombres + ' ' + c.Apellidos AS Nombre,
#     c.Email AS Email,
#     c.Celular AS Celular,
#     v.Fecha AS Fecha_Venta,
#     v.Canal,
#     SUM(v.Venta_Neta) AS Venta
# FROM Ventas_Comerssia.dbo.View_Contacto_Clientes c
# INNER JOIN Ventas_Comerssia.dbo.Ventas_Unificadas v ON c.Cliente = v.Cliente
# WHERE v.fecha >= '2025-07-19'
#     AND v.Fecha <= '2025-07-19'
#     AND v.Venta_Neta > 0
#     AND Canal IN ('Shopify', 'CHATCENTER WEB')
#     AND Linea = 'ALMENDRA'
# GROUP BY  v.Cliente,
#     c.Nombres + ' ' + c.Apellidos,
#     c.Email,
#     c.Celular,
#     v.Fecha,
#     v.Canal
# """

# # Ejecutar y cargar en DataFrame
# df_ventas = pd.read_sql(query, engine)
# df_ventas.head()

In [146]:
# # CONSULTAS CUMPLEAÑOS

# query_data = """
# SELECT DISTINCT v.Cliente, 
# 	cc.Nombres + ' ' +  cc.Apellidos AS Nombre, 
# 	cc.Email, cc.Celular, 
# 	v.Fecha AS Fecha_Venta 
# 	,SUM(Venta_Neta) AS Venta, 
# 	Canal
# FROM Ventas_Comerssia.dbo.Ventas_Unificadas v
# INNER JOIN Ventas_Comerssia.dbo.View_Contacto_Clientes cc ON cc.Cliente = v.Cliente
# WHERE Codigo_Descuento LIKE '%Cump%'
# 	AND Fecha BETWEEN '2025-07-18' AND '2025-07-31'
# 	AND NOT EXISTS (SELECT 1 FROM COMERSSIA.PROVENZAL.dbo.EmpleadosProvenzal e WHERE e.Codigo = v.cliente ) 
# GROUP BY  v.Cliente,
#        cc.Nombres + ' ' +  cc.Apellidos,
#        cc.Email,
#        cc.Celular,
#        v.Fecha,
#        v.Canal
# """

# # Ejecutar y cargar en DataFrame
# df_ventas = pd.read_sql(query_data, engine)
# df_ventas.head()

In [147]:
# Cargar el archivo Excel
df_indigitall = pd.read_excel("Atribucion.xlsx" )

# Seleccionar columnas relevantes
df_campania = df_indigitall[['email', 'campaignName', 'sentAt', 'openedAt', 'clicks']].copy()

# Renombrar columnas
df_campania.rename(columns={
    'email': 'Email',
    'campaignName': 'campania',
    'sentAt': 'fecha_envio',
    'openedAt': 'apertura',
    'clicks': 'clicks'
}, inplace=True)

def extraer_fecha_click(valor):
    try:
        # Convertir string a objeto Python (lista de dicts)
        lista = ast.literal_eval(valor)
        if isinstance(lista, list) and len(lista) > 0 and 'clicked_at' in lista[0]:
            fecha = pd.to_datetime(lista[0]['clicked_at'], utc=True, errors='coerce')
            return fecha.date()  # solo fecha
    except Exception:
        return pd.NaT
    return pd.NaT

df_campania['clicks'] = df_campania['clicks'].apply(extraer_fecha_click)

# Convertir fecha de envío a formato datetime
df_campania['fecha_envio'] = pd.to_datetime(df_campania['fecha_envio'])

# Extraer primera fecha de apertura (si viene como lista en string)
df_campania['apertura'] = df_campania['apertura'].apply(
    lambda x: eval(x)[0] if isinstance(x, str) and x.startswith('[') and len(eval(x)) > 0 else None
)
df_campania['apertura'] = pd.to_datetime(df_campania['apertura'], errors='coerce').dt.tz_localize(None)
df_campania['fecha_envio'] = pd.to_datetime(df_campania['fecha_envio'], errors='coerce').dt.tz_localize(None)

# Dejar solo columnas clave para análisis
df_campania = df_campania[['Email', 'campania', 'fecha_envio', 'apertura', 'clicks']]

In [148]:
# Filtrar solo clientes que abrieron el correo 

df_abiertos = df_campania[df_campania['apertura'].notnull()].copy()
df_clics =  df_campania[df_campania['clicks'].notnull()].copy()

fecha_maxima = pd.to_datetime('2025-07-19').date()
df_abiertos_filtrados = df_abiertos[df_abiertos['apertura'].dt.date <= fecha_maxima]

df_abiertos_filtrados = df_abiertos_filtrados.drop_duplicates(subset='Email', keep='first')
df_clicks_filtrados = df_clics[df_clics['clicks'] <= fecha_maxima]

total_abiertos = df_abiertos_filtrados['Email'].nunique()
total_clicks = df_clicks_filtrados['Email'].nunique()

In [149]:
df_abiertos_filtrados['Email'] = df_abiertos['Email'].str.lower()
df_ventas['Email'] = df_ventas['Email'].str.lower()

# Merge entre ventas y campañas abiertas
df_atribucion = pd.merge(
    df_ventas,
    df_abiertos_filtrados[['Email', 'campania', 'apertura']],
    on='Email',
    how='inner'
)

df_atribucion = df_atribucion.drop_duplicates(subset='Email', keep='first')
filas = len(df_atribucion) # Ver cuántas filas hay realmente

# Sumar el total de ventas atribuidas
total_venta = df_atribucion['Venta'].sum()

print(f"Número de clientes que abrieron hasta {fecha_maxima}: {total_abiertos}")
print(f"Número de clientes que hicieron clics hasta {fecha_maxima}: {total_clicks}")
print(f"Total de venta atribuida: {total_venta:,.0f}")
print(f"Total de filas: {filas}")

df_atribucion.head()

Número de clientes que abrieron hasta 2025-07-19: 2776
Número de clientes que hicieron clics hasta 2025-07-19: 99
Total de venta atribuida: 6,451,613
Total de filas: 22


,Cliente,Nombre,Email,Celular,Fecha_Venta,Canal,Venta,campania,apertura
0,C1020734722,ALEJANDRA VERA CARDONA,alevc71@gmail.com,3176439081,2025-07-19,Shopify,144951.75,Almendras sabado 2.0,2025-07-19 14:20:23
1,C1045676539,MARÍA GONZÁLEZ,mgm.2088@gmail.com,3043915494,2025-07-19,Shopify,141170.40,Almendras sabado 2.0,2025-07-19 15:00:15
2,C21683915,None,verolopez19@hotmail.com,3122243755,2025-07-19,Shopify,732951.67,Almendras sabado 2.0,2025-07-19 15:54:09
3,C28549291,ANDREA PEÑA,andreapenar@yahoo.com,3222169720,2025-07-19,CHATCENTER WEB,222469.42,Almendras sabado 2.0,2025-07-19 18:26:49
4,C30311641,LUZ MARIA GOMEZ,luzmgg2013@gmail.com,3123862094,2025-07-19,Shopify,296835.98,Almendras sabado 2.0,2025-07-19 13:41:23


In [150]:
df_atribucion.to_excel('Ventas.xlsx', index=False)